In [1]:
import anndata as ad
from genome_tools.data.anndata import read_zarr_backed
import pandas as pd
import scipy.sparse as sp
import numpy as np
import h5py

from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os
import h5py
from tqdm import tqdm


In [2]:
p = pd.read_table('/net/seq/data2/projects/nasi4/ENCODE4/dnase+atac-cavs.v1/output/aggregated.extended_annotation.bed')

In [7]:
p.rename(columns={'mean_FMR': 'mean_FDR'}, inplace=True)

In [38]:
required_columns = [
    "#chr",
    "end",
    "ref",
    "alt",
    "group_id",
    "mean_FDR",
    "es_combined",
    "logit_es_combined",
    "min_pval",
    "min_fdr",
    'pval_ref_combined',
    'pval_alt_combined',
    'mean_inverse_mse',
    'mean_cover'
]

variants = p[required_columns]


In [9]:

# =====================================================
# Helper function
# =====================================================

def load_and_process(parquet_file, mapping_file):

    print(f"\nReading {parquet_file}")
    variants = pd.read_parquet(parquet_file)
    print(f"Loaded {len(variants):,} rows")

    sample_map = pd.read_table(mapping_file)

    variants = variants.merge(
        sample_map[["sample_id", "indiv_id"]],
        on="sample_id",
        how="left",
    )

    if "total_counts" not in variants.columns:
        if (
            "ref_counts" in variants.columns
            and "alt_counts" in variants.columns
        ):
            print("Creating total_counts = ref_counts + alt_counts")
            variants["total_counts"] = (
                variants["ref_counts"].fillna(0)
                + variants["alt_counts"].fillna(0)
            )
        else:
            raise ValueError(
                "Missing total_counts and cannot construct it."
            )

    return variants


#load dataset
print("Loading parquet")
variants = load_and_process(
    "/net/seq/data2/projects/nasi4/ENCODE4/dnase+atac-cavs.v1/output/non_aggregated.extended_annotation.parquet",
    "/net/seq/data2/projects/nasi4/ENCODE4/dnase+atac-cavs.v1/atac_dnase_samples_meta.tsv"
)


# Columns required downstream

required_columns = [
    "#chr",
    "end",
    "ref",
    "alt",
    "sample_id",
    "indiv_id",
    "group_id",
    "ref_counts",
    "total_counts",
    "BAD",
    "logit_es",
    "es",
    "inverse_mse",
    "FDR_sample",
]

missing = sorted(set(required_columns) - set(variants.columns))

if missing:
    raise ValueError(
        f"Data is missing required columns:\n{missing}"
    )

variants = variants[required_columns]

print(f"\nLoaded {len(variants):,} variants")
print(f"{variants['group_id'].nunique():,} groups")

Loading DNase

Reading /net/seq/data2/projects/nasi4/ENCODE4/dnase-cavs.v5/output/aggregated.extended_annotation.parquet


FileNotFoundError: [Errno 2] No such file or directory: '/net/seq/data2/projects/nasi4/ENCODE4/dnase-cavs.v5/output/aggregated.extended_annotation.parquet'

In [4]:
#make sure all samples are in the genotype file
print("Loading genotype file")

geno_cols = [
    "#chr",
    "start",
    "end",
    "ref",
    "alt",
    "indiv_id",
    "gt",
]
#### NOTE THIS WILL NEED TO BE REPLACED WITH THE NEW COMBO PHASED GENOTYPE FILE
geno = pd.read_csv(
    "/net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/phased_genotype_dnase_and_atac.bed.gz",
    sep="\t",
    names=geno_cols,
    compression="gzip",
    dtype=str,
)

geno = geno.drop_duplicates()

geno_fixed = pd.DataFrame({
    "#chr": geno.index.astype(str),
    "start": geno["#chr"],
    "end": geno["start"],
    "ref": geno["end"],
    "alt": geno["ref"],
    "indiv_id": geno["alt"],
    "gt": geno["indiv_id"],
})

geno = geno_fixed

key_cols = [
    "#chr",
    "end",
    "ref",
    "alt",
    "indiv_id",
]

for df in [variants, geno]:
    df["#chr"] = df["#chr"].astype(str)
    df["end"] = pd.to_numeric(df["end"])
    df["ref"] = df["ref"].astype(str)
    df["alt"] = df["alt"].astype(str)
    df["indiv_id"] = df["indiv_id"].astype(str)

before_rows = len(variants)
before_groups = variants["group_id"].nunique()

merged = variants.merge(
    geno[key_cols].drop_duplicates(),
    on=key_cols,
    how="left",
    indicator=True,
)

variants = (
    merged[merged["_merge"] == "both"]
    .drop(columns="_merge")
)


print("Genotype filtering summary")
print(f"Rows before : {before_rows:,}")
print(f"Rows after  : {len(variants):,}")
print(f"Rows removed: {before_rows-len(variants):,}")
print()
print(f"Groups before : {before_groups:,}")
print(f"Groups after  : {variants['group_id'].nunique():,}")

Loading genotype file
Genotype filtering summary
Rows before : 126,858,421
Rows after  : 126,840,178
Rows removed: 18,243

Groups before : 67
Groups after  : 67


In [39]:
variants.head()

,#chr,end,ref,alt,group_id,mean_FDR,es_combined,logit_es_combined,min_pval,min_fdr,pval_ref_combined,pval_alt_combined,mean_inverse_mse,mean_cover
0,chr1,794252,C,T,Lung,0.046512,0.448826,-0.291763,0.853084,0.984814,0.904561,0.426542,4.097339,13.666667
1,chr1,794299,C,G,Lung,0.011592,0.445026,-0.313589,1.000000,0.984814,0.889704,0.532205,5.464472,13.500000
2,chr1,794299,C,G,T-cell,0.025000,0.515434,0.087721,1.000000,0.964102,0.625145,0.738521,8.798205,13.000000
3,chr1,804849,C,T,Fibroblast,0.062258,0.515088,0.085755,1.000000,0.917027,0.676277,0.783971,9.767856,16.333333
4,chr1,804849,C,T,Epithelium,0.375000,0.490388,-0.054620,1.000000,0.987118,0.561105,0.571094,9.074323,26.000000


#### for aggregated only adjust columns

In [40]:
variants['sample_id']=variants['group_id']

/tmp/slurm.16546126/ipykernel_1533282/4199345207.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  variants['sample_id']=variants['group_id']


In [41]:
df = variants.rename(
    columns={
        "logit_es_combined": "logit_es",
        "es_combined": "es",
        "min_fdr": "FDR_sample",
        "mean_cover":'total_counts',
        
    }
)

In [42]:
df["sample_id"] = df["group_id"]

# placeholders if unavailable
df["ref_counts"] = np.nan
df["BAD"] = np.nan

In [43]:
df.head()

,#chr,end,ref,alt,group_id,mean_FDR,es,logit_es,min_pval,FDR_sample,pval_ref_combined,pval_alt_combined,mean_inverse_mse,total_counts,sample_id,ref_counts,BAD
0,chr1,794252,C,T,Lung,0.046512,0.448826,-0.291763,0.853084,0.984814,0.904561,0.426542,4.097339,13.666667,Lung,NaN,NaN
1,chr1,794299,C,G,Lung,0.011592,0.445026,-0.313589,1.000000,0.984814,0.889704,0.532205,5.464472,13.500000,Lung,NaN,NaN
2,chr1,794299,C,G,T-cell,0.025000,0.515434,0.087721,1.000000,0.964102,0.625145,0.738521,8.798205,13.000000,T-cell,NaN,NaN
3,chr1,804849,C,T,Fibroblast,0.062258,0.515088,0.085755,1.000000,0.917027,0.676277,0.783971,9.767856,16.333333,Fibroblast,NaN,NaN
4,chr1,804849,C,T,Epithelium,0.375000,0.490388,-0.054620,1.000000,0.987118,0.561105,0.571094,9.074323,26.000000,Epithelium,NaN,NaN


In [44]:
variants = df

In [47]:
required_columns = [
    "#chr",
    "end",
    "ref",
    "alt",
    "sample_id",
    "group_id",
    "ref_counts",
    "total_counts",
    "BAD",
    "logit_es",
    "FDR_sample",
]


### Create nextflow output

In [48]:

h5_out_dir = "/net/seq/data2/projects/mbrannon/6_15_variant_predictons_agg"
os.makedirs(h5_out_dir, exist_ok=True)

prediction_manifest = []

zarr_anndata = "/home/mbrannon/tmp/group_embeddings.h5ad"
fasta_file = "/net/seq/data/genomes/human/GRCh38/noalts/GRCh38_no_alts.fa"
model_config = "/net/seq/data2/projects/mbrannon/variant_model_paper/2026_06_08_happy_bose/run_config.yaml"
# genotype_file = "/net/seq/data2/projects/mbrannon/variant_model_paper/genotypes/phased_genotype_dnase_and_atac.bed.gz"
genotype_file = None
checkpoint = "/net/seq/data2/projects/mbrannon/variant_model_paper/2026_06_08_happy_bose/checkpoints/epoch=14-step=58021-val_loss=3.44.ckpt"

# =====================================================
# Rename map
# =====================================================

rename_map = {
    "#chr": "chrom",
    "end": "pos",
}

# =====================================================
# Determine string vs numeric columns automatically
# =====================================================

string_columns = []

numeric_columns = []

for col in required_columns:

    if col == "group_id":
        continue

    if (
        variants[col].dtype == object
        or pd.api.types.is_string_dtype(variants[col])
    ):
        string_columns.append(col)
    else:
        numeric_columns.append(col)

# =====================================================
# Write H5 files
# =====================================================

variants = variants.sort_values("group_id")

groups = variants.groupby("group_id", sort=False)

for group_id, df in tqdm(
    groups,
    total=variants["group_id"].nunique(),
    desc="Writing groups",
):

    safe_group = (
        str(group_id)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
    )

    outfile = os.path.join(
        h5_out_dir,
        f"{safe_group}_variants.h5",
    )

    with h5py.File(outfile, "w") as f:

        for col in string_columns:

            save_name = rename_map.get(col, col)

            f.create_dataset(
                save_name,
                data=np.asarray(
                    df[col].astype(str),
                    dtype=h5py.string_dtype("utf-8"),
                ),
                compression="gzip",
            )

        for col in numeric_columns:

            save_name = rename_map.get(col, col)

            f.create_dataset(
                save_name,
                data=df[col].to_numpy(),
                compression="gzip",
            )

    prediction_manifest.append(
        {
            "prefix": safe_group,
            "zarr_anndata": zarr_anndata,
            "fasta_file": fasta_file,
            "model_config": model_config,
            "genotype_file": genotype_file,
            "checkpoint": checkpoint,
            "dhs_dataset": outfile,
        }
    )

# =====================================================
# Save manifest
# =====================================================

manifest_df = pd.DataFrame(prediction_manifest)

manifest_out = os.path.join(
    h5_out_dir,
    "prediction_input_manifest.tsv",
)

manifest_df.to_csv(
    manifest_out,
    sep="\t",
    index=False,
)

print()
print(f"Finished writing {len(prediction_manifest):,} H5 files")
print(f"Manifest saved to:\n{manifest_out}")

Writing groups: 100%|██████████| 63/63 [00:45<00:00,  1.40it/s]


Finished writing 63 H5 files
Manifest saved to:
/net/seq/data2/projects/mbrannon/6_15_variant_predictons_agg/prediction_input_manifest.tsv


In [ ]:
#how to run not nextflow
srun -p hpcg-test --gres=gpu:1 --cpus-per-task=8 --mem=64G --time=08:00:00 python /home/mbrannon/.local/src/vinson/predict_all_variants.py \
/net/seq/data2/projects/mbrannon/vinson/3_9_dataset_20totalcount1p5bad_3epoch.h5ad \
/net/seq/data/genomes/human/GRCh38/noalts/GRCh38_no_alts.fa \
/net/seq/data2/projects/mbrannon/variant_model_test/2026_03_30_silly_kirch/checkpoints/epoch=8-step=5781-val_loss=2.93.ckpt \
/net/seq/data2/projects/mbrannon/variant_model_test/2026_03_30_silly_kirch/run_config.yaml \
--genotype_file /home/mbrannon/genotype_no_multiallelic.tsv.gz \
--h5_file /net/seq/data2/projects/mbrannon/variant_predictons/B-cell_variants.h5 \
--batch_size 64 \
--num_workers 8 \
--output /net/seq/data2/projects/mbrannon/variant_predictons/bcelltst.npy